# Análise de Multiplicação de Matrizes

Este notebook permite treinar com diferentes tamanhos de matrizes, comparar execução serial e paralela, e gerar gráficos e tabelas de desempenho.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter

plt.style.use('seaborn-v0_8')
%matplotlib inline

In [ ]:
def generate_random_matrices(n, low=-10, high=10, seed=None):
    if seed is not None:
        np.random.seed(seed)
    a = np.random.randint(low, high + 1, size=(n, n))
    b = np.random.randint(low, high + 1, size=(n, n))
    return a, b

def serial_multiply(a, b):
    n = a.shape[0]
    result = np.zeros((n, b.shape[1]), dtype=a.dtype)
    for i in range(n):
        for j in range(b.shape[1]):
            for k in range(a.shape[1]):
                result[i, j] += a[i, k] * b[k, j]
    return result

def _multiply_row(row, b):
    return np.dot(row, b)

def _multiply_block(block, b):
    return np.vstack([np.dot(row, b) for row in block])

def parallel_multiply(a, b, workers=4):
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(_multiply_row, a[i], b) for i in range(a.shape[0])]
        result = np.vstack([future.result() for future in futures])
    return result

def distributed_multiply(a, b, workers=4):
    blocks = np.array_split(a, workers, axis=0)
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(_multiply_block, block, b) for block in blocks]
        result = np.vstack([future.result() for future in futures])
    return result

def example_matrices():
    a = np.array([
        [1, 0, 2, 3],
        [4, -1, 1, 5],
        [-2, -3, -4, 2],
        [-1, 2, 0, 0],
    ])
    b = np.array([
        [-1, 1, 2, -3],
        [-5, -4, 2, -2],
        [3, -1, 0, 2],
        [1, 0, 4, 5],
    ])
    return a, b

def benchmark(sizes, repeats=3, workers=4, seed=42):
    rows = []
    for n in sizes:
        serial_times = []
        parallel_times = []
        distributed_times = []
        for _ in range(repeats):
            a, b = generate_random_matrices(n, seed=seed)
            t0 = perf_counter()
            serial_multiply(a, b)
            serial_times.append(perf_counter() - t0)

            t1 = perf_counter()
            parallel_multiply(a, b, workers=workers)
            parallel_times.append(perf_counter() - t1)

            t2 = perf_counter()
            distributed_multiply(a, b, workers=workers)
            distributed_times.append(perf_counter() - t2)

        rows.append({
            'size': n,
            'serial_ms': np.mean(serial_times) * 1000,
            'parallel_ms': np.mean(parallel_times) * 1000,
            'distributed_ms': np.mean(distributed_times) * 1000,
            'parallel_speedup': np.mean(serial_times) / np.mean(parallel_times),
            'distributed_speedup': np.mean(serial_times) / np.mean(distributed_times),
        })
    return pd.DataFrame(rows)

In [ ]:
sizes = [20, 50, 100, 200]
df = benchmark(sizes, repeats=3, workers=4, seed=123)
df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df['size'], df['serial_ms'], marker='o', label='Serial')
ax.plot(df['size'], df['parallel_ms'], marker='o', label='Paralelo (4 workers)')
ax.plot(df['size'], df['distributed_ms'], marker='o', label='Distribuído (blocos)')
ax.set_xlabel('Tamanho da matriz (n)')
ax.set_ylabel('Tempo médio (ms)')
ax.set_title('Comparação de desempenho serial, paralelo e distribuído')
ax.legend()
ax.grid(True)
plt.show()

fig2, ax2 = plt.subplots(figsize=(8, 5))
indices = np.arange(len(df['size']))
width = 0.35
ax2.bar(indices - width/2, df['parallel_speedup'], width=width, label='Speedup paralelo')
ax2.bar(indices + width/2, df['distributed_speedup'], width=width, label='Speedup distribuído')
ax2.set_xticks(indices)
ax2.set_xticklabels(df['size'])
ax2.set_xlabel('Tamanho da matriz (n)')
ax2.set_ylabel('Speedup em relação ao serial')
ax2.set_title('Ganho de velocidade: paralelo vs distribuído')
ax2.legend()
ax2.grid(True)
plt.show()

In [ ]:
df.to_csv('benchmark_results.csv', index=False)
print('Benchmark salvo em benchmark_results.csv')
df

In [ ]:
a, b = example_matrices()
print('Matriz A exemplo:')
print(a)
print('\nMatriz B exemplo:')
print(b)
print('\nResultado esperado:')
print(np.dot(a, b))